# Anomaly detection: MVTec AD на Kaggle

Перед запуском включите **Accelerator → GPU** и подключите Dataset `ipythonx/mvtec-ad`. DTD понадобится только для DRAEM. Результаты сохраняются в `/kaggle/working/experiments`.

In [ ]:
%cd /kaggle/working
![ -d anomaly-detection-reproduction ] || GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/calmik0lya/anomaly-detection-reproduction.git
%cd /kaggle/working/anomaly-detection-reproduction
!git pull --ff-only
!pip install -q -r requirements-kaggle.txt

## Проверка GPU
PyTorch 2.7.1 CUDA 12.6 установлен специально для Tesla P100 (`sm_60`).

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('Architectures:', torch.cuda.get_arch_list())
print('GPU test:', torch.tensor([1.0], device='cuda'))

## Восстановление результатов предыдущей сессии

После **Save Version** можно подключить output предыдущей версии как Dataset. Ячейка найдёт сохранённые experiment-папки в `/kaggle/input` и вернёт их в рабочую папку для `--resume`. При первом запуске она ничего не копирует.

In [ ]:
from pathlib import Path
import shutil
destination = Path('/kaggle/working/experiments')
destination.mkdir(parents=True, exist_ok=True)
restored = 0
for metrics_file in Path('/kaggle/input').glob('**/experiments/*__*__*/metrics.json'):
    source_run = metrics_file.parent
    target_run = destination / source_run.name
    if not target_run.exists():
        shutil.copytree(source_run, target_run)
        restored += 1
print('Восстановлено экспериментов:', restored)

## Проверка путей
MVTec обязателен для всех моделей. DTD проверяется отдельно и нужен только перед запуском DRAEM.

In [ ]:
from pathlib import Path
import yaml
paths = yaml.safe_load(Path('configs/paths/kaggle.yaml').read_text())
mvtec = Path(paths['mvtec_path'])
dtd = Path(paths['dtd_images_path'])
print('mvtec_path:', mvtec, 'OK' if mvtec.exists() else 'НЕ НАЙДЕН')
print('dtd_images_path:', dtd, 'OK' if dtd.exists() else 'НЕ ПОДКЛЮЧЁН (нужен только DRAEM)')
if not mvtec.exists():
    raise FileNotFoundError('Подключите Kaggle Dataset ipythonx/mvtec-ad')

## Быстрая проверка команды без обучения

In [ ]:
!python run.py --config configs/models/patchcore.yaml --paths configs/paths/kaggle.yaml --category bottle --device cuda --dry-run

## Контрольный реальный запуск
Сначала запускаем только PatchCore на bottle и проверяем папку с конфигом, метриками и весами.

In [ ]:
!python run.py --config configs/models/patchcore.yaml --paths configs/paths/kaggle.yaml --category bottle --device cuda --output-dir /kaggle/working/experiments
!find /kaggle/working/experiments -maxdepth 3 -type f | sort

## Пакетный запуск с продолжением
Начинайте с одной-двух моделей. После завершения сохраните Version с output. В новой сессии подключите предыдущий output, выполните ячейку восстановления и повторите команду: `--resume` пропустит готовые эксперименты.

In [ ]:
!python sweep.py --models patchcore padim --paths configs/paths/kaggle.yaml --device cuda --output-dir /kaggle/working/experiments --resume --continue-on-error

Для следующих запусков замените модели на `stfpm`, затем `simplenet`, затем `draem`. DRAEM запускайте последним.